# Assignment 1 — QANet (High-EM Training Notebook)

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. This notebook is configured for **full-data training** to maximize dev F1 / EM.

> Before training, switch Colab to **GPU runtime**. Full-data QANet is too memory-heavy for a normal CPU runtime.

> If you only want a quick smoke test, switch Section 1 and Section 2 back to `download_mini()` and `train-mini.json`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026_fixed"


In [ ]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en_core_web_sm


---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [ ]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

---
## Section 1 — Download Full Data *(one-time setup)*

Downloads the full SQuAD v1.1 train/dev set and the full GloVe 840B 300d vectors into `_data/`.

> This is the recommended setup if your goal is to improve EM substantially. The mini dataset is useful only for pipeline smoke tests.


In [ ]:
from Tools.download import download

download(data_dir="_data")


---
## Section 2 — Preprocess Full Data *(one-time setup)*

Tokenises the full SQuAD JSON files, builds word/char vocabularies from the full GloVe file, and writes padded index tensors to `_data/`.

> Re-run only if you change `para_limit`, `ques_limit`, `char_limit`, or related preprocessing limits.


In [ ]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-v1.1.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.840B.300d.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)


---
## Section 3 — Train for High EM

Trains QANet on the full SQuAD v1.1 training set and saves the **best checkpoint by dev EM/F1** to `_model_full/model.pt`.

> Standard Colab should start with `batch_size=4`. If memory is still tight, reduce to `2`. Increase above `4` only if the runtime clearly has enough memory.


In [ ]:
from TrainTools.train import train

results = train(
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model_full",
    log_dir         = "_log_full",
    num_steps       = 12000,
    checkpoint      = 1000,
    batch_size      = 4,
    seed            = 42,
    optimizer_name  = "adam",
    scheduler_name  = "cosine",
    learning_rate   = 1e-3,
    loss_name       = "qa_nll",
    norm_name       = "layer_norm",
    test_num_batches = -1,
    max_answer_len  = 30,
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")


---
## Section 4 — Evaluate Best Checkpoint

Loads the saved best checkpoint and runs inference on the **full dev set**.


In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz          = "_data/dev.npz",
    word_emb_json    = "_data/word_emb.json",
    char_emb_json    = "_data/char_emb.json",
    dev_eval_json    = "_data/dev_eval.json",
    save_dir         = "_model_full",
    log_dir          = "_log_full",
    ckpt_name        = "model.pt",
    test_num_batches = -1,
    max_answer_len   = 30,
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")
